In [3]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Flatten, GlobalAveragePooling2D, Concatenate, Dropout
from tensorflow.keras.applications import InceptionV3
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Load handcrafted features
data = pd.read_csv("../Dataset/Features/signature_features.csv")
labels = data["label"].values
handcrafted_features = data.iloc[:, 2:].values  # Skip filename and label columns

# Load images for CNN
image_size = (299, 299)  # InceptionV3 default input size
image_paths = [f"/Users/pasan_diksura/Documents/SoftwareDevelopment/Projects/Signature-Forgery-Detection-System/SignatureForgeryDetectionSystem/MachineLearning/Dataset/Processing/AugmentedDataset/{'Genuine' if label == 0 else 'Forged'}/{file}" for file, label in zip(data["filename"], labels)]

# Load and preprocess images
def load_images(image_paths):
    images = []
    for path in image_paths:
        img = tf.keras.preprocessing.image.load_img(path, target_size=image_size)
        img = tf.keras.preprocessing.image.img_to_array(img)
        img = tf.keras.applications.inception_v3.preprocess_input(img)
        images.append(img)
    return np.array(images)

images = load_images(image_paths)

# Split dataset
X_train_img, X_test_img, X_train_handcrafted, X_test_handcrafted, y_train, y_test = train_test_split(
    images, handcrafted_features, labels, test_size=0.2, random_state=42, stratify=labels)

# CNN Feature Extractor (InceptionV3)
inception_base = InceptionV3(weights=None, include_top=False, input_shape=(299, 299, 3))
cnn_output = GlobalAveragePooling2D()(inception_base.output)

# LSTM for sequence modeling
lstm_input = Input(shape=(X_train_handcrafted.shape[1], 1))  # Handcrafted features
lstm_layer = LSTM(128, return_sequences=False)(lstm_input)

# Combine CNN and LSTM
merged = Concatenate()([cnn_output, lstm_layer])
dense_layer = Dense(128, activation='relu')(merged)
dropout_layer = Dropout(0.5)(dense_layer)
output_layer = Dense(1, activation='sigmoid')(dropout_layer)

# Define hybrid model
model = Model(inputs=[inception_base.input, lstm_input], outputs=output_layer)

# Compile model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train model
model.fit(
    [X_train_img, X_train_handcrafted.reshape(-1, X_train_handcrafted.shape[1], 1)], y_train,
    validation_data=([X_test_img, X_test_handcrafted.reshape(-1, X_test_handcrafted.shape[1], 1)], y_test),
    epochs=25, batch_size=32)

# Save model
model.save("../Model/signature_forgery_detection_model.h5")

print("Model training completed and saved!")


Epoch 1/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 131s 3s/step - accuracy: 0.7535 - loss: 0.5584 - val_accuracy: 0.4054 - val_loss: 0.7397
Epoch 2/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 114s 3s/step - accuracy: 0.8649 - loss: 0.3301 - val_accuracy: 0.5946 - val_loss: 1.9219
Epoch 3/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 118s 3s/step - accuracy: 0.8338 - loss: 0.3660 - val_accuracy: 0.5946 - val_loss: 1.9208
Epoch 4/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 117s 3s/step - accuracy: 0.9106 - loss: 0.2278 - val_accuracy: 0.5946 - val_loss: 4.3587
Epoch 5/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 118s 3s/step - accuracy: 0.9388 - loss: 0.1670 - val_accuracy: 0.5946 - val_loss: 3.3689
Epoch 6/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 117s 3s/step - accuracy: 0.9459 - loss: 0.1656 - val_accuracy: 0.5946 - val_loss: 2.0553
Epoch 7/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 117s 3s/step - accuracy: 0.9329 - loss: 0.1645 - val_accuracy: 0.5946 - val_loss: 2.9187
Epoch 8/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 117s 3s/step - accuracy: 0.9409 - loss: 0.1450 - val_accuracy: 0.5946 - v

Model training completed and saved!


In [4]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Predict on test set
y_pred_probs = model.predict([X_test_img, X_test_handcrafted.reshape(-1, X_test_handcrafted.shape[1], 1)])
y_pred = (y_pred_probs > 0.5).astype(int)  # Convert probabilities to binary predictions

# Compute evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

# Print results
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")
print("Confusion Matrix:")
print(conf_matrix)


10/10 ━━━━━━━━━━━━━━━━━━━━ 9s 862ms/step
Accuracy: 0.7905
Precision: 0.9833
Recall: 0.4917
F1-score: 0.6556
Confusion Matrix:
[[175   1]
 [ 61  59]]
